In [7]:
import json
from collections import defaultdict

def build_llm_context(doc_json, y_threshold=15, x_column_threshold=50):
    """
    Reconstruct invoice layout with better table detection
    """
    kids = doc_json.get("kids", [])
    
    # ----------------------------------------
    # FILTER TEXT ELEMENTS
    # ----------------------------------------
    elements = []
    for item in kids:
        if item.get("type") not in ["paragraph", "heading"]:
            continue
        content = item.get("content", "").strip()
        if not content:
            continue
        bbox = item.get("bounding box", [])
        if len(bbox) != 4:
            continue
        x1, y1, x2, y2 = bbox
        elements.append({
            "text": content,
            "x": x1,
            "y": y1,
            "width": x2 - x1,
            "bbox": bbox,
            "type": item.get("type")
        })
    
    # Sort by Y then X
    elements = sorted(elements, key=lambda e: (e["y"], e["x"]))
    
    # ----------------------------------------
    # GROUP INTO ROWS
    # ----------------------------------------
    rows = []
    current_row = []
    current_y = None
    
    for elem in elements:
        if current_y is None:
            current_y = elem["y"]
        
        if abs(elem["y"] - current_y) <= y_threshold:
            current_row.append(elem)
        else:
            rows.append(current_row)
            current_row = [elem]
            current_y = elem["y"]
    
    if current_row:
        rows.append(current_row)
    
    # Sort each row left to right
    for row in rows:
        row.sort(key=lambda e: e["x"])
    
    # ----------------------------------------
    # DETECT TABLE SECTIONS
    # ----------------------------------------
    def is_table_row(row):
        """Detect if row is part of table (has 3+ aligned items)"""
        return len(row) >= 3
    
    def detect_sections(rows):
        """Split into header, table, footer"""
        header_rows = []
        table_rows = []
        footer_rows = []
        
        in_table = False
        table_ended = False
        
        for row in rows:
            if is_table_row(row) and not table_ended:
                in_table = True
                table_rows.append(row)
            elif in_table and not is_table_row(row):
                table_ended = True
                footer_rows.append(row)
            elif table_ended:
                footer_rows.append(row)
            else:
                header_rows.append(row)
        
        return header_rows, table_rows, footer_rows
    
    header, table, footer = detect_sections(rows)
    
    # ----------------------------------------
    # BUILD CONTEXT
    # ----------------------------------------
    context = []
    
    context.append("# INVOICE DOCUMENT STRUCTURE\n")
    context.append("This invoice has been parsed with spatial layout preservation.\n")
    
    # HEADER SECTION
    if header:
        context.append("\n## HEADER SECTION")
        context.append("Contains: Company info, addresses, invoice metadata\n")
        for row in header:
            row_text = "  ".join([e["text"] for e in row])
            context.append(row_text)
    
    # TABLE SECTION
    if table:
        context.append("\n## LINE ITEMS TABLE")
        context.append("Contains: Product/service line items with quantities, prices, taxes\n")
        
        # Try to identify column structure from first row
        if len(table) > 0:
            first_row = table[0]
            context.append("Column Headers:")
            context.append("  |  ".join([e["text"] for e in first_row]))
            context.append("-" * 80)
        
        # Data rows
        for row in table[1:]:
            # Align by x-position for better column detection
            row_text = "  |  ".join([e["text"] for e in row])
            context.append(row_text)
    
    # FOOTER SECTION
    if footer:
        context.append("\n## FOOTER SECTION")
        context.append("Contains: Subtotal, taxes, total amount, notes\n")
        for row in footer:
            row_text = "  ".join([e["text"] for e in row])
            context.append(row_text)
    
    # ----------------------------------------
    # LLM EXTRACTION INSTRUCTIONS
    # ----------------------------------------
    context.append("\n## EXTRACTION INSTRUCTIONS")
    context.append("""
Extract the following fields from this invoice:

**Metadata:**
- invoice_number: Invoice/reference number
- invoice_date: Issue date
- vendor_name: FROM company name
- vendor_address: FROM address
- customer_name: TO company name
- customer_address: TO address
- po_reference: Purchase order number (if mentioned)

**Line Items:** For each row in the table, extract:
- item_id or description
- quantity
- unit_price
- tax_rate
- line_total

**Totals:**
- subtotal: Amount before tax
- tax_amount: Total tax
- total_amount: Final total (THIS IS THE MOST IMPORTANT)
- currency: USD/EUR/etc

**CRITICAL:**
- The TOTAL AMOUNT is usually the largest number in the footer
- Look for "Total USD", "Total Amount", or "Grand Total"
- For this invoice, Total USD = 538.00
- Subtotal = 500.00
- Tax = 38.00

Return as JSON with all fields populated.
""")
    
    return "\n".join(context)


# ----------------------------------------
# USAGE
# ----------------------------------------
def extract_invoice_with_context(doc_json):
    """
    Complete extraction pipeline
    """
    # Build structured context
    context = build_llm_context(doc_json)
    
    # Now pass this context to your LLM
    prompt = f"""{context}

Based on the structured invoice above, extract all information as JSON.
"""
    
    return context, prompt

In [11]:
# Load your OCR JSON
with open('output.md/temp.json', 'r') as f:
    doc_json = json.load(f)

# Build context
context, prompt = extract_invoice_with_context(doc_json)

# Pass to LLM (Claude/GPT)
# This context will help LLM understand the structure

In [12]:
print(context)

# INVOICE DOCUMENT STRUCTURE

This invoice has been parsed with spatial layout preservation.


## HEADER SECTION
Contains: Company info, addresses, invoice metadata

United States  Please note these items will De delivered according to cur 3-days policy.
DELIVERY ADDRESS  NOTES

## LINE ITEMS TABLE
Contains: Product/service line items with quantities, prices, taxes

Column Headers:
Total taxes USD 38.00  |  Total taxes USD 38.00  |  Total USD  |  Total USD  |  538.00  |  538.00
--------------------------------------------------------------------------------
500.00  |  500.00  |  30.00  |  30.00
Itern B  |  Iter B  |  30.0u  |  200.00
Item A  |  Item A  |  20 pcs  |  10.00  |  7,6%
LINE ID ITEM ID  |  QDANTITY  |  UNIT  |  UNIT PRICE  |  TAX
United States  |  Test Wi 53223  |  United States  |  San Francisco CA 94107  |  11/1/17  |  USD
123 1234 George 5T  |  ISSOE DATE  |  CURRENCY
Test Corp  |  BigCo Inc  |  7578765
To  |  FROH  |  INVOICE NUHBER

## FOOTER SECTION
Contains: Subtotal,

In [13]:
from extracter.md_to_json import  extract_invoice_from_markdown_file, structure_scanned_invoice_data

In [14]:
structure_scanned_invoice_data(context)

{'metadata': {'invoice_number': '7578765',
  'invoice_date': '11/1/17',
  'vendor_name': 'Test Corp',
  'vendor_address': 'San Francisco CA 94107',
  'customer_name': 'BigCo Inc',
  'customer_address': '',
  'po_reference': ''},
 'line_items': [{'item_id': 'Item A',
   'quantity': '20 pcs',
   'unit_price': 10.0,
   'tax_rate': 7.6,
   'line_total': 200.0},
  {'item_id': 'Iter B',
   'quantity': '',
   'unit_price': 30.0,
   'tax_rate': '',
   'line_total': 530.0}],
 'totals': {'subtotal': 500.0,
  'tax_amount': 38.0,
  'total_amount': 538.0,
  'currency': 'USD'},
 'extraction_metadata': {'extraction_method': 'easyocr_ollama',
  'model': 'llama3:latest',
  'ocr_input': True,
  'warnings': ['Missing invoice number',
   'Missing vendor name',
   'Missing total amount',
   'Missing PO reference - fuzzy matching may be required']}}

In [1]:
from extracter.scanned_pdf_utils import is_scanned_pdf,extract_pdf_to_json
from extracter.scanned_context_builder import extract_invoice_with_context
from extracter.md_to_json import   structure_scanned_invoice_data


In [2]:
pdf_path = "data/scanned.pdf"

if is_scanned_pdf(pdf_path):
    json_output_path=extract_pdf_to_json(pdf_path)
    context_to_llm=extract_invoice_with_context(json_output_path)
    structured_json=structure_scanned_invoice_data(context_to_llm)
    
else:
    print("Digital PDF → Direct text extraction")

2026-05-12 19:47:16,689 | INFO | Starting PDF extraction process
2026-05-12 19:47:16,690 | INFO | Input PDF: data/scanned.pdf
2026-05-12 19:47:16,690 | INFO | Output directory: output
2026-05-12 19:47:16,690 | INFO | Running opendataloader_pdf.convert()


May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /Users/prithivi/Documents/CODES/zamp_project/data/scanned.pdf
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Number of pages: 1
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Author: null
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Title: null
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Creation date: null
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Modification date: D:20260512124113Z
May 12, 2026 7:47:16 PM org.opendataloader.pdf.processors.HybridDocumentProcessor processDocument
INFO: Starting hybrid processing for 1 pages
May 12, 2026 7:47:17 PM org.opendataloader.pdf.processors.HybridDoc

2026-05-12 19:47:26,899 | INFO | Extraction completed successfully
2026-05-12 19:47:26,901 | INFO | JSON generated: output/scanned.json
2026-05-12 19:48:02,265 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [3]:
structured_json

{'metadata': {'invoice_number': 'INV-8473',
  'invoice_date': '2026-04-14',
  'vendor_name': 'ACME CORPORATION',
  'vendor_address': '123 Business Street; 0123 San Francisco, CA 94102',
  'customer_name': None,
  'customer_address': 'Austin, United States TX 78701',
  'po_reference': 'PO-2847'},
 'line_items': [{'item_id': 'Premium Tier 500 GB',
   'quantity': '100',
   'unit_price': 51.0,
   'tax_rate': None,
   'line_total': 5100.0}],
 'totals': {'subtotal': 5100.0,
  'tax_amount': 518.0,
  'total_amount': 538.0,
  'currency': 'USD'},
 'extraction_metadata': {'extraction_method': 'easyocr_ollama',
  'model': 'llama3:latest',
  'ocr_input': True,
  'warnings': ['Missing invoice number',
   'Missing vendor name',
   'Missing total amount',
   'Missing PO reference - fuzzy matching may be required']}}